# 🎬 Netflix Content Library Analysis — Exploratory Data Analysis (Python)

**Author:** Data Analysis Portfolio Project
**Dataset:** [Netflix Movies and TV Shows](https://www.kaggle.com/datasets/shivamb/netflix-shows) (Kaggle) — `netflix_titles.csv`, 8,807 titles
**Tools:** Python, Pandas, Matplotlib, Seaborn

---

## 📌 Project Overview

This notebook is the **Python/pandas companion** to a SQL-based analytical portfolio project (`Database_analysis.sql`), where the same Netflix catalog was normalized into a relational schema (`titles`, `genres`, `title_genres`, `actors`, `title_actors`) and queried in MySQL using joins, subqueries, `CASE` expressions, and window functions (`RANK() OVER (PARTITION BY ...)`).

This notebook reproduces and visualizes the **same 20 business insights** end-to-end, but working directly from the flat CSV file using `pandas` for the "relational" logic (splitting multi-value fields, grouping, ranking) and `matplotlib` / `seaborn` for static, publication-quality visualizations — the kind you'd put in a report or a GitHub README.

### Objective
Understand Netflix's content strategy across five dimensions:

| Section | Focus | Insights |
|---|---|---|
| **A** | Content Trends | Library growth, movie/TV split, freshness, seasonality |
| **B** | Genre Analysis | Top genres, genre evolution, genre mix by type/era |
| **C** | Country & Geography | Content-producing countries, movie/TV ratio by country, co-productions |
| **D** | Actor & Director Analysis | Most prolific directors/actors, collaborations |
| **E** | Ratings & Duration | Content rating distribution, runtime trends, season counts |

### Why pandas mirrors SQL here
Every `GROUP BY` becomes a `.groupby()`, every `JOIN` becomes a `.merge()` (or, since genres/cast are comma-separated strings in the raw CSV rather than normalized tables, a `.str.split(',').explode()`), and every `RANK() OVER (PARTITION BY ...)` becomes `.groupby().rank()` or `.groupby().head()`. Where relevant, each code cell is annotated with the SQL query it corresponds to.


## 1. Setup & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ---- Global plot styling ----
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titlepad"] = 14

# Netflix brand-inspired palette
NETFLIX_RED = "#E50914"
NETFLIX_DARK = "#221f1f"
PALETTE = ["#E50914", "#B20710", "#F5F5F1", "#564d4d", "#831010", "#a3a3a3"]
sns.set_palette(sns.color_palette(PALETTE))

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 2. Load the Data

In [ ]:
df = pd.read_csv("netflix_titles.csv")

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

In [ ]:
df.info()

## 3. Data Quality Assessment

Before any analysis, we check completeness and look for structural issues — the same due-diligence step that, in the SQL version, happened during schema design and `NOT NULL` handling.

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})[missing > 0]

**Observations:**
- `director` is missing for ~30% of titles, `cast` for ~9%, `country` for ~9% — all handled by excluding nulls at the point of each specific analysis (matching the SQL `WHERE column IS NOT NULL` pattern), not by dropping rows globally.
- `date_added`, `rating`, and `duration` are missing for a handful of rows only.

**Structural data-quality issue found:** a small number of rows have their `duration` value shifted into the `rating` column (e.g. `rating = "74 min"`). This is a known quirk of this dataset (a source-file column shift) and must be repaired before any rating or duration analysis.

In [ ]:
shifted_mask = df["rating"].isin(["74 min", "84 min", "66 min"])
print(f"Rows with shifted columns: {shifted_mask.sum()}")
df.loc[shifted_mask, ["show_id", "type", "title", "rating", "duration"]]

In [ ]:
# Repair: move the mis-placed duration value back to `duration`, null out the bogus rating
df.loc[shifted_mask, "duration"] = df.loc[shifted_mask, "rating"]
df.loc[shifted_mask, "rating"] = np.nan
print("Repaired.")

## 4. Feature Engineering (the "normalization" step)

The SQL version normalizes the raw CSV into `titles`, `genres` (via `title_genres`), and `actors` (via `title_actors`) tables. Here we replicate that logic in pandas:

1. Parse `date_added` into a proper datetime, and derive `year_added` / `month_added`.
2. Split `duration` into `duration_minutes` (Movies) and `duration_seasons` (TV Shows) — equivalent to the two typed columns in the SQL `titles` table.
3. Derive `primary_country` — SQL used `SUBSTRING_INDEX(country, ',', 1)` to take the first listed country; we do the same with `.str.split(',').str[0]`.
4. Explode `listed_in` (genres) into a long-format table — equivalent to the `title_genres` join table.
5. Explode `cast` into a long-format table — equivalent to the `title_actors` join table.

In [ ]:
# 4.1 Dates
df["date_added"] = pd.to_datetime(df["date_added"].str.strip(), format="%B %d, %Y", errors="coerce")
df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month_name()

# 4.2 Duration split (type-dependent, exactly like two separate typed SQL columns)
duration_num = df["duration"].str.extract(r"(\d+)").astype(float)[0]
df["duration_minutes"] = np.where(df["type"] == "Movie", duration_num, np.nan)
df["duration_seasons"] = np.where(df["type"] == "TV Show", duration_num, np.nan)

# 4.3 Primary country  ==  SUBSTRING_INDEX(country, ',', 1)
df["primary_country"] = df["country"].str.split(",").str[0].str.strip()

# 4.4 Genre "join table"  ==  title_genres JOIN genres
genres_long = (
    df[["show_id", "type", "release_year", "date_added", "year_added"]]
    .join(df["listed_in"].str.split(",").rename("genre_name"))
    .explode("genre_name")
)
genres_long["genre_name"] = genres_long["genre_name"].str.strip()
genres_long = genres_long.dropna(subset=["genre_name"])

# 4.5 Cast "join table"  ==  title_actors JOIN actors
cast_long = (
    df[["show_id", "type", "director", "country"]]
    .join(df["cast"].str.split(",").rename("actor_name"))
    .explode("actor_name")
)
cast_long["actor_name"] = cast_long["actor_name"].str.strip()
cast_long = cast_long.dropna(subset=["actor_name"])

print("titles          :", df.shape)
print("genres_long     :", genres_long.shape, " (long-format, one row per title-genre pair)")
print("cast_long       :", cast_long.shape, " (long-format, one row per title-actor pair)")

---
# Section A — Content Trends

*(SQL reference: Section A, queries A1–A4)*

### A1. Movie vs TV Show Split

In [ ]:
type_counts = df["type"].value_counts()
type_pct = (type_counts / len(df) * 100).round(2)

fig, ax = plt.subplots(figsize=(7, 6))
wedges, texts, autotexts = ax.pie(
    type_counts, labels=type_counts.index, autopct="%1.1f%%",
    colors=[NETFLIX_RED, NETFLIX_DARK], startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 12, "fontweight": "bold", "color": "white"}
)
ax.set_title("Movie vs. TV Show Split\n(% of Full Netflix Catalog)")
plt.tight_layout()
plt.show()

print(type_counts)
print(type_pct)

**Insight A1** — Movies dominate the library: **6,131 titles (69.6%)** vs. **2,676 TV Shows (30.4%)**. Netflix's catalog is roughly a 7:3 movie-heavy split — TV Shows remain the minority format by a wide margin, even though streaming is often culturally associated with binge-worthy series.

### A2. Content Added by Year & Type (Library Growth Trend)

In [ ]:
growth = (
    df.dropna(subset=["date_added"])
    .groupby(["year_added", "type"])
    .size()
    .unstack(fill_value=0)
)
growth = growth.loc[(growth.index >= 2008) & (growth.index <= 2021)]  # trim sparse early years

fig, ax = plt.subplots(figsize=(12, 6))
growth.plot(kind="bar", stacked=True, ax=ax, color=[NETFLIX_RED, NETFLIX_DARK], width=0.75)
ax.set_title("Titles Added to Netflix per Year, by Type")
ax.set_xlabel("Year Added")
ax.set_ylabel("Number of Titles")
ax.legend(title="Type")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Share of TV Shows within each year's additions
tv_share = (growth["TV Show"] / growth.sum(axis=1) * 100).round(1)
print("TV Show share of yearly additions (%):")
print(tv_share.tail(6))

**Insight A2** — Explosive growth from 2016–2019, then a pullback. Titles added per year jumped from 429 (2016) to 1,188 (2017) to 1,649 (2018) to a peak of 2,016 (2019) — more than a 4x increase in three years. Both 2020 (1,879) and 2021 (1,498) declined from the 2019 peak, likely reflecting pandemic-era production shutdowns slowing content acquisition. TV Shows' share of yearly additions fluctuated rather than rising steadily — it dipped from 41.0% (2016) to a low of 25.0% (2018) as Netflix's movie-acquisition binge outpaced TV, then climbed back to 33.7% by 2021. So while movies still dominate the *total* catalog, TV Shows have been recovering share of what's newly added in the most recent years.

### A3. Release-to-Addition Gap by Type (How Fresh Is Netflix's Content?)

In [ ]:
gap = df.dropna(subset=["date_added"]).copy()
gap["years_gap"] = gap["year_added"] - gap["release_year"]
avg_gap = gap.groupby("type")["years_gap"].mean().round(2)

fig, ax = plt.subplots(figsize=(8, 5.5))
bars = ax.bar(avg_gap.index, avg_gap.values, color=[NETFLIX_RED, NETFLIX_DARK], width=0.5)
ax.bar_label(bars, fmt="%.2f yrs", fontsize=12, fontweight="bold", padding=6)
ax.set_title("Average Gap Between Release Year and Netflix Add Date")
ax.set_ylabel("Average Years")
ax.set_ylim(0, avg_gap.max() * 1.25)
plt.tight_layout()
plt.show()

print(avg_gap)

**Insight A3** — TV Shows are added far "fresher" than Movies. The average gap between release year and Netflix add date is **5.73 years for Movies** vs. just **2.30 years for TV Shows** — less than half. Netflix functions more like a back-catalog library for film, but stays closer to "current" for series, consistent with licensing fresh seasons quickly while movies rely more heavily on older catalog deals.

### A4. Content Added by Month (Release Seasonality)

In [ ]:
month_order = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]
monthly = df.dropna(subset=["date_added"])["month_added"].value_counts().reindex(month_order)

fig, ax = plt.subplots(figsize=(12, 6))
colors = [NETFLIX_RED if v == monthly.max() else (NETFLIX_DARK if v == monthly.min() else "#a3a3a3") for v in monthly]
bars = ax.bar(monthly.index, monthly.values, color=colors)
ax.set_title("Titles Added by Month (Release Seasonality, All Years Combined)")
ax.set_xlabel("Month")
ax.set_ylabel("Number of Titles")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

print(monthly.sort_values(ascending=False))

**Insight A4** — **July (827)** and **December (813)** are the top two months for adding content — both align with school holidays and year-end viewing spikes. **February** is the clear low point (563 titles), nearly 32% below July — the shortest month also gets the least new content, whether due to the calendar-day effect or a deliberate lighter release slate.

---
# Section B — Genre Analysis

*(SQL reference: Section B, queries B1–B4)*

### B1. Top Genres by Title Count

In [ ]:
top_genres = genres_long["genre_name"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 6.5))
sns.barplot(x=top_genres.values, y=top_genres.index, hue=top_genres.index,
            palette=[NETFLIX_RED]*3 + [NETFLIX_DARK]*7, legend=False, ax=ax)
ax.set_title("Top 10 Genres by Title Count (Titles Carry Multiple Genre Tags)")
ax.set_xlabel("Number of Titles")
ax.set_ylabel("")
for i, v in enumerate(top_genres.values):
    ax.text(v + 15, i, f"{v:,}", va="center", fontsize=10)
plt.tight_layout()
plt.show()

**Insight B1** — "International Movies" is the single biggest genre tag (**2,752 titles**), ahead of Dramas (2,427) and Comedies (1,674). Since titles carry multiple genre tags, this reflects how heavily Netflix leans on "International" as a catch-all label for its non-US-origin catalog — it appears on nearly 1 in 3 titles overall.

### B2. Genre Evolution — Average Release Year per Genre

In [ ]:
genre_evo = (
    genres_long.groupby("genre_name")
    .agg(title_count=("show_id", "size"), avg_release_year=("release_year", "mean"))
)
genre_evo = genre_evo[genre_evo["title_count"] >= 100].sort_values("avg_release_year")
genre_evo["avg_release_year"] = genre_evo["avg_release_year"].round(1)

fig, ax = plt.subplots(figsize=(10, 9))
colors = plt.cm.RdGy_r(np.linspace(0.15, 0.85, len(genre_evo)))
ax.barh(genre_evo.index, genre_evo["avg_release_year"] - genre_evo["avg_release_year"].min(),
        left=genre_evo["avg_release_year"].min(), color=colors)
ax.set_xlim(genre_evo["avg_release_year"].min() - 1, genre_evo["avg_release_year"].max() + 1)
ax.set_title("Genre Evolution: Average Release Year by Genre (min. 100 titles)")
ax.set_xlabel("Average Release Year")
plt.tight_layout()
plt.show()

genre_evo.sort_values("avg_release_year").head(3)

**Insight B2** — "Action & Adventure" is the oldest-skewing major genre (average release year **2009.5**), while "TV Dramas" and "International TV Shows" skew newest (2017.2 and 2017.0). This lines up with A1/A3 — TV content on Netflix is systematically newer than movie content, and that pattern holds at the genre level too, not just the type level.

### B3. Genre Mix: Top 5 Genres for Movies vs. TV Shows

In [ ]:
# RANK() OVER (PARTITION BY type ORDER BY COUNT(*) DESC)  ==  groupby().rank(method='min')
genre_by_type = genres_long.groupby(["type", "genre_name"]).size().reset_index(name="title_count")
genre_by_type["rnk"] = genre_by_type.groupby("type")["title_count"].rank(method="min", ascending=False)
top5_by_type = genre_by_type[genre_by_type["rnk"] <= 5].sort_values(["type", "rnk"])

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)
for ax, t, color in zip(axes, ["Movie", "TV Show"], [NETFLIX_RED, NETFLIX_DARK]):
    sub = top5_by_type[top5_by_type["type"] == t].sort_values("title_count")
    ax.barh(sub["genre_name"], sub["title_count"], color=color)
    ax.set_title(f"Top 5 Genres — {t}")
    ax.set_xlabel("Number of Titles")
fig.suptitle("Movies and TV Shows Have Almost No Genre Overlap in Their Top 5", fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

**Insight B3** — Movies and TV Shows have almost no genre overlap in their top 5. Movies are led by International Movies (2,752), Dramas (2,427), Comedies (1,674), Documentaries (869), and Action & Adventure (859). TV Shows are led by International TV Shows (1,351), TV Dramas (763), TV Comedies (581), Crime TV Shows (470), and Kids' TV (451) — completely separate genre vocabularies, matching how Netflix tags TV-specific vs. Movie-specific categories.

### B4. Early vs. Recent Genre Focus — Pre-2018 vs. 2018+ Additions

In [ ]:
era_genres = genres_long.dropna(subset=["year_added"]).copy()
era_genres["era"] = np.where(era_genres["year_added"] < 2018, "Pre-2018", "2018 & Later")

era_grp = era_genres.groupby(["era", "genre_name"]).size().reset_index(name="title_count")
era_grp["rnk"] = era_grp.groupby("era")["title_count"].rank(method="min", ascending=False)
top5_era = era_grp[era_grp["rnk"] <= 5].sort_values(["era", "rnk"])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, era, color in zip(axes, ["Pre-2018", "2018 & Later"], [NETFLIX_DARK, NETFLIX_RED]):
    sub = top5_era[top5_era["era"] == era].sort_values("title_count")
    ax.barh(sub["genre_name"], sub["title_count"], color=color)
    ax.set_title(era)
    ax.set_xlabel("Number of Titles")
fig.suptitle("Top-5 Genres: Pre-2018 vs. 2018-and-Later Additions", fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

**Insight B4** — The top genre is stable over time (International Movies leads both eras), but "Action & Adventure" breaks into the recent top-5 (741 titles, 2018+) while it wasn't a pre-2018 leader — suggesting a deliberate content-strategy shift toward action content as Netflix scaled up its catalog after 2018.

---
# Section C — Country & Geography

*(SQL reference: Section C, queries C1–C4)*

### C1. Top Content-Producing Countries (Primary Country)

In [ ]:
top_countries = df["primary_country"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 6.5))
colors = [NETFLIX_RED if i < 2 else NETFLIX_DARK for i in range(len(top_countries))]
sns.barplot(x=top_countries.values, y=top_countries.index, hue=top_countries.index,
            palette=colors, legend=False, ax=ax)
ax.set_title("Top 10 Content-Producing Countries (Primary Country of Origin)")
ax.set_xlabel("Number of Titles")
ax.set_ylabel("")
for i, v in enumerate(top_countries.values):
    ax.text(v + 15, i, f"{v:,}", va="center", fontsize=10)
plt.tight_layout()
plt.show()

**Insight C1** — The **United States (3,211)** and **India (1,008)** are the two dominant content origins by far, together accounting for over half of all titles with a known country. The United Kingdom (628) is a distant third. This is a heavily US/India-concentrated catalog, not a globally even spread across the ~86 countries represented.

### C2. Movie/TV Ratio by Top Countries

In [ ]:
top8 = df["primary_country"].value_counts().head(8).index
ratio = (
    df[df["primary_country"].isin(top8)]
    .groupby(["primary_country", "type"]).size()
    .unstack(fill_value=0)
)
ratio_pct = ratio.div(ratio.sum(axis=1), axis=0) * 100
ratio_pct = ratio_pct.loc[ratio_pct["Movie"].sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(11, 6.5))
ratio_pct.plot(kind="barh", stacked=True, ax=ax, color=[NETFLIX_RED, NETFLIX_DARK])
ax.set_title("Movie vs. TV Show Mix by Top 8 Countries (%)")
ax.set_xlabel("% of Country's Titles")
ax.set_ylabel("")
ax.legend(title="Type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

ratio_pct.round(1)

**Insight C2** — India is the most movie-skewed major country (**92.0% Movie**, 927 vs. 81 TV Shows) — almost no TV Show presence. South Korea (22.3% movie) and Japan (32.8% movie) sit at the exact opposite end — both are TV/anime-heavy catalogs. The US (73.6% movie) and UK (60.8% movie) fall in between, closer to the global average — this isn't a "movies dominate everywhere" pattern, it varies sharply by country of origin.

### C3. Top Genre per Top-5 Country

In [ ]:
top5_countries = df["primary_country"].value_counts().head(5).index

country_genres = (
    df[df["primary_country"].isin(top5_countries)][["show_id", "primary_country", "listed_in"]]
    .join(df["listed_in"].str.split(",").rename("genre_name"))
    .explode("genre_name")
)
country_genres["genre_name"] = country_genres["genre_name"].str.strip()

cg = country_genres.groupby(["primary_country", "genre_name"]).size().reset_index(name="title_count")
cg["rnk"] = cg.groupby("primary_country")["title_count"].rank(method="min", ascending=False)
top3_cg = cg[cg["rnk"] <= 3].sort_values(["primary_country", "rnk"])

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=top3_cg, y="primary_country", x="title_count", hue="genre_name",
            dodge=True, ax=ax, palette="Reds_r")
ax.set_title("Top 3 Genres for Each of the Top 5 Content-Producing Countries")
ax.set_xlabel("Number of Titles")
ax.set_ylabel("")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, title="Genre")
plt.tight_layout()
plt.show()

**Insight C3** — India's #1 genre is International Movies (845) — its own domestic output is tagged as "international" from Netflix's (US-centric) catalog perspective. Japan's top genre is International TV Shows (144), closely followed by Anime Series (136) — anime is a near-equal pillar of its catalog, not a side category. The UK stands out as the only top-5 country whose #1 genre is a country-specific tag (British TV Shows, 216) rather than a generic Drama/International label — a distinct enough catalog to earn its own tag.

### C4. Single-Country vs. Co-Production Titles

In [ ]:
has_country = df.dropna(subset=["country"]).copy()
has_country["production_type"] = np.where(
    has_country["country"].str.contains(","), "Co-production (2+ countries)", "Single country"
)
prod_counts = has_country["production_type"].value_counts()
prod_pct = (prod_counts / len(has_country) * 100).round(2)

fig, ax = plt.subplots(figsize=(7, 6))
ax.pie(prod_counts, labels=prod_counts.index, autopct="%1.1f%%",
       colors=[NETFLIX_DARK, NETFLIX_RED], startangle=90,
       wedgeprops={"edgecolor": "white", "linewidth": 2},
       textprops={"fontsize": 11, "fontweight": "bold", "color": "white"})
ax.set_title("Single-Country vs. Co-Produced Titles")
plt.tight_layout()
plt.show()

print(prod_counts)
print(prod_pct)

**Insight C4** — Co-productions are a minority but not negligible: **1,320 titles (16.55%)** list 2+ countries, vs. 83.45% single-country. Among co-produced titles, the US appears as a partner country most often (393 titles) — even when it's not the sole country of origin, American studios/distributors are frequently involved behind the scenes.

---
# Section D — Actor & Director Analysis

*(SQL reference: Section D, queries D1–D4)*

### D1. Top Directors by Title Count

In [ ]:
top_directors = df.dropna(subset=["director"])["director"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 6.5))
sns.barplot(x=top_directors.values, y=top_directors.index, hue=top_directors.index,
            palette=[NETFLIX_RED]*3 + [NETFLIX_DARK]*7, legend=False, ax=ax)
ax.set_title("Top 10 Directors by Title Count")
ax.set_xlabel("Number of Titles")
ax.set_ylabel("")
for i, v in enumerate(top_directors.values):
    ax.text(v + 0.15, i, str(v), va="center", fontsize=10)
plt.tight_layout()
plt.show()

**Insight D1** — Rajiv Chilaka leads with 19 titles, almost all Indian animated/children's content — ahead of Raúl Campos & Jan Suter (18, a directing duo credited jointly) and Suhas Kadav (16). Notably, none of the most "famous" Hollywood names top the list by volume — the top spots go to high-output regional/children's-content directors, not prestige filmmakers, since this ranks by title count rather than acclaim.

### D2. Top Actors by Title Count

In [ ]:
top_actors = cast_long["actor_name"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 6.5))
sns.barplot(x=top_actors.values, y=top_actors.index, hue=top_actors.index,
            palette=[NETFLIX_RED]*1 + [NETFLIX_DARK]*9, legend=False, ax=ax)
ax.set_title("Top 10 Actors by Title Count")
ax.set_xlabel("Number of Titles")
ax.set_ylabel("")
for i, v in enumerate(top_actors.values):
    ax.text(v + 0.4, i, str(v), va="center", fontsize=10)
plt.tight_layout()
plt.show()

**Insight D2** — Anupam Kher tops the actor list with 43 titles, and Bollywood names dominate the top 10 overall (Shah Rukh Khan 35, Naseeruddin Shah 32, Akshay Kumar 30, Om Puri 30, Amitabh Bachchan 28) alongside Japanese voice actors (Takahiro Sakurai 32, Yuki Kaji 29) from anime series. This mirrors C1's finding — India's outsized share of the catalog surfaces again here, this time in cast credits rather than country tags.

### D3. Director-Actor Frequent Collaborations

In [ ]:
collab_base = (
    df.dropna(subset=["director"])[["show_id", "director"]]
    .merge(cast_long[["show_id", "actor_name"]], on="show_id", how="inner")
)
collabs = (
    collab_base.groupby(["director", "actor_name"]).size()
    .reset_index(name="collab_count")
    .sort_values("collab_count", ascending=False)
    .head(10)
)
collabs["pair"] = collabs["director"] + "  +  " + collabs["actor_name"]

fig, ax = plt.subplots(figsize=(11, 6.5))
ax.barh(collabs["pair"], collabs["collab_count"], color=NETFLIX_RED)
ax.invert_yaxis()
ax.set_title("Top 10 Director-Actor Collaborations by Shared Title Count")
ax.set_xlabel("Number of Shared Titles")
plt.tight_layout()
plt.show()

**Insight D3** — Rajiv Chilaka's regular voice-cast (Julie Tejwani, Jigna Bhardwaj, Rajesh Kava — 17 titles each) shows a tight, repeat production team typical of long-running animated series, not one-off film casts. This is a completely different collaboration pattern from the Movies/Dramas world (no single Hollywood director-actor pair even approaches these numbers) — recurring animated series inflate collaboration counts far more than feature films do.

### D4. Top-5 Directors — Type & Country Focus

In [ ]:
top5_directors = df.dropna(subset=["director"])["director"].value_counts().head(5).index

d4 = df[df["director"].isin(top5_directors)].copy()
d4_summary = (
    d4.groupby("director")
    .agg(
        primary_country=("primary_country", lambda s: s.mode().iat[0] if not s.mode().empty else np.nan),
        title_count=("show_id", "size"),
        movies=("type", lambda s: (s == "Movie").sum()),
        tv_shows=("type", lambda s: (s == "TV Show").sum()),
    )
    .sort_values("title_count", ascending=False)
)
d4_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
d4_summary[["movies", "tv_shows"]].plot(kind="barh", stacked=True, ax=ax, color=[NETFLIX_RED, NETFLIX_DARK])
ax.set_title("Top 5 Directors: Movie vs. TV Show Output")
ax.set_xlabel("Number of Titles")
ax.set_ylabel("")
ax.legend(title="Type")
plt.tight_layout()
plt.show()

**Insight D4** — All 5 top directors are 100% Movie directors — none of them have a single TV Show credit in the dataset. Rajiv Chilaka and Suhas Kadav are both India-based (19 and 16 titles respectively), while Raúl Campos/Jan Suter (Mexico) and Marcus Raboy/Jay Karas (United States) round out the list — the most prolific directors cluster in comedy-special and animated-film production, a very different profile from prestige feature directing.

---
# Section E — Ratings & Duration

*(SQL reference: Section E, queries E1–E4)*

### E1. Content Rating Distribution

In [ ]:
rating_counts = df.dropna(subset=["rating"])["rating"].value_counts()
rating_pct = (rating_counts / rating_counts.sum() * 100).round(2)

fig, ax = plt.subplots(figsize=(11, 6.5))
colors = [NETFLIX_RED if i < 2 else NETFLIX_DARK for i in range(len(rating_counts))]
sns.barplot(x=rating_counts.values, y=rating_counts.index, hue=rating_counts.index,
            palette=colors, legend=False, ax=ax)
ax.set_title("Content Rating Distribution (All Titles)")
ax.set_xlabel("Number of Titles")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

rating_pct.head(5)

**Insight E1** — TV-MA is the single largest rating (**3,207 titles, 36.4%**) — more than a third of the entire catalog is mature-audience content. TV-MA + TV-14 combined make up 61% of all titles — Netflix's catalog leans heavily toward teen/adult content rather than family-friendly fare; G/TV-Y/TV-Y7 family-safe ratings combined make up well under 10% of the library.

### E2. Average Movie Duration Trend by Decade

In [ ]:
def decade_bucket(y):
    if y < 2000: return "Pre-2000"
    if y <= 2009: return "2000s"
    if y <= 2019: return "2010s"
    return "2020+"

movies = df[(df["type"] == "Movie") & df["duration_minutes"].notna()].copy()
movies["decade"] = movies["release_year"].apply(decade_bucket)
decade_order = ["Pre-2000", "2000s", "2010s", "2020+"]
decade_avg = movies.groupby("decade")["duration_minutes"].mean().reindex(decade_order).round(1)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(decade_avg.index, decade_avg.values, color=[NETFLIX_DARK, NETFLIX_DARK, NETFLIX_RED, NETFLIX_RED])
ax.bar_label(bars, fmt="%.1f min", fontsize=11, fontweight="bold", padding=6)
ax.set_title("Average Movie Runtime by Release Decade — A Steady Decline")
ax.set_ylabel("Average Duration (minutes)")
ax.set_ylim(0, decade_avg.max() * 1.2)
plt.tight_layout()
plt.show()

**Insight E2** — Movies have gotten steadily shorter every decade: Pre-2000 averaged **114.97 min**, dropping to 112.11 min (2000s), 96.91 min (2010s), and just **93.64 min (2020+)** — a ~20% reduction from pre-2000 levels. This aligns with the broader streaming-era trend toward shorter runtimes optimized for attention spans and algorithmic recommendation, not a one-off dip.

### E3. TV Show Seasons Distribution

In [ ]:
seasons = df[df["type"] == "TV Show"]["duration_seasons"].dropna()
season_counts = seasons.value_counts().sort_index()
one_season_pct = round((seasons == 1).sum() / len(seasons) * 100, 1)

fig, ax = plt.subplots(figsize=(11, 6.5))
colors = [NETFLIX_RED if s == 1 else NETFLIX_DARK for s in season_counts.index]
ax.bar(season_counts.index.astype(int).astype(str), season_counts.values, color=colors)
ax.set_title(f"TV Show Seasons Distribution  ({one_season_pct}% of Shows Have Only 1 Season)")
ax.set_xlabel("Number of Seasons")
ax.set_ylabel("Number of Shows")
plt.tight_layout()
plt.show()

**Insight E3** — 67% of all TV Shows on Netflix have only 1 season (1,793 of 2,676) — the vast majority never get renewed past their first season, or are limited series by design. Shows with 5+ seasons make up a small tail (fewer than 5% combined) — long-running multi-season series are the exception, not the norm, on this platform.

### E4. Duration Extremes by Genre — Longest & Shortest Average Movie Runtime

In [ ]:
movie_genres = (
    df[df["type"] == "Movie"][["show_id", "duration_minutes", "listed_in"]]
    .dropna(subset=["duration_minutes"])
    .join(df["listed_in"].str.split(",").rename("genre_name"))
    .explode("genre_name")
)
movie_genres["genre_name"] = movie_genres["genre_name"].str.strip()

mgd = (
    movie_genres.groupby("genre_name")
    .agg(movie_count=("show_id", "size"), avg_duration_minutes=("duration_minutes", "mean"))
)
mgd = mgd[mgd["movie_count"] >= 20]
mgd["avg_duration_minutes"] = mgd["avg_duration_minutes"].round(1)

longest = mgd.sort_values("avg_duration_minutes", ascending=False).head(5)
shortest = mgd.sort_values("avg_duration_minutes", ascending=True).head(5)
extremes = pd.concat([longest.assign(category="Longest"), shortest.assign(category="Shortest")])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, cat, color in zip(axes, ["Longest", "Shortest"], [NETFLIX_RED, NETFLIX_DARK]):
    sub = extremes[extremes["category"] == cat].sort_values("avg_duration_minutes")
    ax.barh(sub.index, sub["avg_duration_minutes"], color=color)
    ax.set_title(f"{cat} Average Runtime")
    ax.set_xlabel("Average Duration (minutes)")
fig.suptitle("Movie Runtime Extremes by Genre (min. 20 movies)", fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

**Insight E4** — Classic Movies run longest on average (118.6 min), followed by Action & Adventure (113.5 min) and Dramas (113.1 min) — genres built around extended narrative or spectacle. Stand-Up Comedy is the shortest major genre (67.3 min) after the tiny "Movies" catch-all tag — comedy specials are structurally built to run under 90 minutes, nearly 40% shorter than the overall movie average (99.6 min).

---
# 📋 Summary of Findings

| # | Insight | Key Number |
|---|---|---|
| A1 | Movies dominate the catalog | 69.6% Movie / 30.4% TV Show |
| A2 | Library additions grew ~4x from 2016–2019, then declined | 429 → 2,016 titles/yr |
| A3 | TV Shows are added far fresher than Movies | 2.30 yrs vs. 5.73 yrs gap |
| A4 | July & December are peak addition months | 827 / 813 titles |
| B1 | "International Movies" is the top genre tag | 2,752 titles |
| B2 | Action & Adventure is the oldest-skewing genre | avg. release year 2009.5 |
| B3 | Movies and TV Shows have almost no top-genre overlap | — |
| B4 | Action & Adventure rose into the top-5 post-2018 | 741 titles (2018+) |
| C1 | US and India dominate content origin | 3,211 / 1,008 titles |
| C2 | India is the most movie-skewed major country | 92.0% Movie |
| C3 | UK is the only top-5 country with a country-specific top genre | British TV Shows, 216 |
| C4 | Co-productions are a meaningful minority | 16.55% of titled content |
| D1 | Rajiv Chilaka is the most prolific director by volume | 19 titles |
| D2 | Bollywood actors dominate the top-10 by title count | Anupam Kher, 43 titles |
| D3 | Animated-series voice casts drive the top collaborations | 17 shared titles |
| D4 | All top-5 directors are 100% Movie directors | 0 TV credits |
| E1 | TV-MA is the largest content rating | 36.4% of catalog |
| E2 | Movie runtimes have shrunk ~20% since pre-2000 | 115.0 → 93.6 min |
| E3 | Most TV Shows never get renewed past season 1 | 67% single-season |
| E4 | Stand-Up Comedy is the shortest major genre | 67.3 min avg. |

## 🔑 Strategic Takeaways

1. **Content acquisition strategy differs sharply by format.** Netflix treats movies as a back-catalog library (older, longer average gap to add) and TV as a current-affairs product (fresher, faster-growing share of yearly additions).
2. **The catalog is not globally even** — it's anchored by the US and India, with each country showing a distinct movie/TV mix (India almost all-movie, Japan/South Korea TV-heavy).
3. **The library skews toward mature audiences** (TV-MA + TV-14 = 61%), and runtimes have compressed over time — both consistent with an attention-economy content strategy.
4. **High title-count directors/actors are dominated by high-output regional and children's/animated content**, not prestige single releases — a reminder that "most prolific" and "most acclaimed" are very different rankings.

---

*This notebook is the pandas/matplotlib/seaborn counterpart to `Database_analysis.sql`, which performs the equivalent analysis in MySQL using joins, subqueries, and window functions on a normalized relational schema. Both notebooks derive identical headline numbers from the same source dataset, cross-validating the two approaches.*
